**3 Model Training PySpark + Scikit-learn baseline (single node)**

11) Train 4 MLlib models

In [ ]:
# Define models
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=200)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10)
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=50)  # binary by default; use OvR for multiclass
ovr_gbt = OneVsRest(classifier=gbt, featuresCol="features", labelCol="label")

# Fit pipelines without tuning
trained_pipes = {}

for name, clf in [("LR", lr), ("RF", rf), ("DT", dt), ("OvR_GBT", ovr_gbt)]:
    pipe = Pipeline(stages=[indexer, cust_trans, assembler, scaler, pca, clf])
    print(f"Training {name} ...")
    trained_pipes[name] = pipe.fit(train_df_ckpt)

print("Trained models:", list(trained_pipes.keys()))


12) Hyperparameter tuning (CrossValidator) for at least 1 model

In [ ]:
# Logistic Regression model with base settings
lr_tune = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50
)

# Build full ML pipeline:
# 1) Convert label to indexed format
# 2) Create custom nutrient score
# 3) Assemble features
# 4) Scale features
# 5) Apply PCA (dimensionality reduction)
# 6) Train Logistic Regression
lr_pipe = Pipeline(stages=[indexer, cust_trans, assembler, scaler, pca, lr_tune])

# Define hyperparameter search space
# regParam controls regularisation strength
# elasticNetParam controls L1 vs L2 balance
paramGrid = (ParamGridBuilder()
             .addGrid(lr_tune.regParam, [0.1, 0.01])
             .addGrid(lr_tune.elasticNetParam, [0.0, 0.5])
             .build())

# CrossValidator performs k-fold validation
# numFolds=2 keeps computation manageable
# parallelism=2 allows parallel training of parameter combinations
cv = CrossValidator(
    estimator=lr_pipe,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator.setMetricName("f1"),
    numFolds=2,
    parallelism=2
)

print("Training tuned LR (CV)...")

# Open Spark UI BEFORE CrossValidator
ui_url = spark.sparkContext.uiWebUrl
print("Spark UI internal URL:", ui_url)

m = re.search(r":(\d+)", ui_url or "")
port = int(m.group(1)) if m else 4040

print("Opening Spark UI on port:", port)
output.serve_kernel_port_as_window(port)

# Train model using cross-validation
# The best parameter combination is selected automatically
best_lr_model = cv.fit(train_df_ckpt)

13) Serialization

In [ ]:
best_model_path = "best_food_model_spark"
best_lr_model.bestModel.write().overwrite().save(best_model_path)
print("Saved Spark best model to:", best_model_path)

14) scikit-learn baseline (single node) + pickle

In [ ]:
# Single-node baseline comparison using sklearn
# This section trains Logistic Regression and Random Forest on a small pandas sample to compare distributed Spark ML
# performance with a traditional single-machine approach.

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.ensemble import RandomForestClassifier as SkRF


# Convert a small Spark sample to pandas
# Keeping the fraction small avoids memory issues in Colab
sample_pd = (curated_df
             .sample(withReplacement=False, fraction=0.01, seed=42)
             .select("nova_group", *feature_cols)
             .toPandas()
             .dropna())

# Separate features (X) and target labels (y)
y = sample_pd["nova_group"].astype(int).values
X = sample_pd[feature_cols].astype(float).values

# Split into training and testing sets
# Stratified split preserves class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rows = []

# -------------------------------------------------------
# sklearn Logistic Regression (with feature scaling)
# -------------------------------------------------------

# Pipeline ensures scaling is applied before training
lr_pipe = SkPipeline([
    ("scaler", StandardScaler()),
    ("clf", SkLR(max_iter=500, n_jobs=-1, multi_class="auto"))
])

# Measure training time
t0 = time.time()
lr_pipe.fit(X_train, y_train)
t_train = time.time() - t0

# Measure inference time
t0 = time.time()
pred = lr_pipe.predict(X_test)
t_infer = time.time() - t0

# Save trained model
with open("sklearn_lr_baseline.pkl", "wb") as f:
    pickle.dump(lr_pipe, f)

# Store evaluation results
rows.append({
    "Model": "sklearn_LR",
    "accuracy": accuracy_score(y_test, pred),
    "weightedPrecision": precision_score(y_test, pred, average="weighted", zero_division=0),
    "weightedRecall": recall_score(y_test, pred, average="weighted", zero_division=0),
    "weightedF1": f1_score(y_test, pred, average="weighted", zero_division=0),
    "train_time_sec": t_train,
    "inference_time_sec": t_infer,
    "n_train": len(y_train),
    "n_test": len(y_test)
})

# -------------------------------------------------------
# sklearn Random Forest (scaling not required)
# -------------------------------------------------------

sk_rf = SkRF(n_estimators=200, n_jobs=-1, random_state=42)

# Measure training time
t0 = time.time()
sk_rf.fit(X_train, y_train)
t_train = time.time() - t0

# Measure inference time
t0 = time.time()
pred2 = sk_rf.predict(X_test)
t_infer = time.time() - t0

# Save trained model
with open("sklearn_rf_baseline.pkl", "wb") as f:
    pickle.dump(sk_rf, f)

# Store evaluation results
rows.append({
    "Model": "sklearn_RF",
    "accuracy": accuracy_score(y_test, pred2),
    "weightedPrecision": precision_score(y_test, pred2, average="weighted", zero_division=0),
    "weightedRecall": recall_score(y_test, pred2, average="weighted", zero_division=0),
    "weightedF1": f1_score(y_test, pred2, average="weighted", zero_division=0),
    "train_time_sec": t_train,
    "inference_time_sec": t_infer,
    "n_train": len(y_train),
    "n_test": len(y_test)
})

# Convert results to DataFrame and export for reporting / Tableau
baseline_df = pd.DataFrame(rows)
baseline_df.to_csv("sklearn_baseline_metrics.csv", index=False)

print("Saved: sklearn_baseline_metrics.csv")
baseline_df